# Notebook 1: Data Exploration & Regime Analysis
**Project:** Crude Oil Price Forecasting & Trading Signal Generation  
**Goal:** Understand Brent/WTI price dynamics, identify market regimes, and validate data quality

## 1. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})
print("Libraries loaded.")

## 2. Load Price Data

In [ ]:
from pathlib import Path

def load_or_generate():
    brent_path = Path('../data/raw/brent_ohlcv.parquet')
    if brent_path.exists():
        brent = pd.read_parquet(brent_path)
        wti   = pd.read_parquet('../data/raw/wti_ohlcv.parquet')
        print(f"Loaded real data: {len(brent):,} trading days")
    else:
        print("Data not found. Run: python src/ingestion/download_data.py")
        print("Generating synthetic data for notebook demonstration...")
        from src.ingestion.download_data import generate_synthetic_prices, _generate_synthetic_inventory, generate_synthetic_news_sentiment
        generate_synthetic_prices('2005-01-01')
        _generate_synthetic_inventory()
        generate_synthetic_news_sentiment()
        brent = pd.read_parquet(brent_path)
        wti   = pd.read_parquet('../data/raw/wti_ohlcv.parquet')
    return brent, wti

brent, wti = load_or_generate()
print(f"Brent: {brent.index[0].date()} → {brent.index[-1].date()}")
print(f"Price range: ${brent['close'].min():.1f} — ${brent['close'].max():.1f}/bbl")
brent.tail(3)

## 3. Long-Term Price History with Key Events

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

axes[0].plot(brent.index, brent['close'], color='#378ADD', linewidth=1.0, label='Brent')
axes[0].plot(wti.index, wti['close'], color='#EF9F27', linewidth=0.8, alpha=0.7, label='WTI')
axes[0].set_ylabel('Price ($/bbl)')
axes[0].set_title('Brent & WTI Crude Oil — 20-Year Price History')
axes[0].legend()

# Key events
events = [
    ('2008-07-11', 'All-time high\n$147', 'red'),
    ('2009-01-15', '2009\nCrash', 'red'),
    ('2014-11-28', '2014 OPEC\nGlut', 'red'),
    ('2020-04-21', 'COVID\nCrash', 'red'),
    ('2022-03-08', 'Ukraine\nWar', 'orange'),
]
for date_str, label, color in events:
    try:
        ax_date = pd.Timestamp(date_str)
        price = brent['close'].asof(ax_date) if ax_date in brent.index or True else None
        if price and not pd.isna(price):
            axes[0].annotate(label, xy=(ax_date, price),
                           xytext=(0, 25), textcoords='offset points',
                           fontsize=7, ha='center', color=color,
                           arrowprops=dict(arrowstyle='->', color=color, lw=0.8))
    except: pass

spread = brent['close'] - wti['close'].reindex(brent.index).ffill()
axes[1].fill_between(spread.index, spread, alpha=0.6, color='#1D9E75',
                     label='Brent-WTI Spread')
axes[1].axhline(spread.mean(), color='#888', linestyle='--', linewidth=0.8,
                label=f'Mean spread: ${spread.mean():.2f}')
axes[1].set_ylabel('Spread ($/bbl)')
axes[1].set_title('Brent-WTI Spread (Positive = Brent Premium)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../docs/images/price_history.png', bbox_inches='tight')
plt.show()

## 4. Return Distribution Analysis

In [ ]:
returns = brent['close'].pct_change().dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(returns, bins=80, color='#378ADD', edgecolor='white', density=True, alpha=0.8)
import scipy.stats as stats
x = np.linspace(returns.min(), returns.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, returns.mean(), returns.std()),
             'r--', linewidth=1.5, label='Normal fit')
axes[0].set_title('Daily Return Distribution')
axes[0].set_xlabel('Daily Return')
axes[0].legend()

rolling_vol = returns.rolling(60).std() * np.sqrt(252) * 100
axes[1].plot(rolling_vol.index, rolling_vol, color='#EF9F27', linewidth=0.8)
axes[1].fill_between(rolling_vol.index, rolling_vol, alpha=0.3, color='#EF9F27')
axes[1].set_title('60-Day Rolling Annualised Volatility (%)')
axes[1].set_ylabel('Volatility (%)')

axes[2].bar(range(5), returns.abs().nlargest(5).values, color='#E24B4A', edgecolor='white')
axes[2].set_xticklabels([str(d.date()) for d in returns.abs().nlargest(5).index], rotation=45, ha='right', fontsize=7)
axes[2].set_title('Largest Single-Day Moves')
axes[2].set_ylabel('|Return|')

plt.tight_layout()
plt.savefig('../docs/images/return_distribution.png', bbox_inches='tight')
plt.show()
print(f"Skewness: {returns.skew():.3f} (negative = left tail)")
print(f"Kurtosis: {returns.kurtosis():.3f} (>3 = fat tails)")
print(f"Annual vol: {returns.std()*np.sqrt(252)*100:.1f}%")

## 5. Market Regime Detection

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Features for regime detection
feat = pd.DataFrame({
    'vol': returns.rolling(20).std() * np.sqrt(252),
    'trend': brent['close'].pct_change(60),
    'rsi': None,
}, index=brent.index)

# Simple RSI
delta = brent['close'].diff()
gain = delta.clip(lower=0).ewm(com=13, adjust=False).mean()
loss = (-delta).clip(lower=0).ewm(com=13, adjust=False).mean()
rs = gain / loss.clip(lower=1e-10)
feat['rsi'] = (100 - 100/(1+rs)) / 100

feat = feat.dropna()
scaler = StandardScaler()
X = scaler.fit_transform(feat)

# K-means regime clustering (3 regimes: bull, bear, sideways)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
regimes = km.fit_predict(X)
regime_series = pd.Series(regimes, index=feat.index)

# Label regimes by mean return
regime_returns = {}
for r in range(3):
    mask = regime_series == r
    regime_returns[r] = returns.reindex(feat.index[mask]).mean()
regime_labels = {r: ('Bull' if v == max(regime_returns.values()) else
                     'Bear' if v == min(regime_returns.values()) else 'Sideways')
                 for r, v in regime_returns.items()}

colors = {'Bull': '#1D9E75', 'Bear': '#E24B4A', 'Sideways': '#EF9F27'}
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(brent.index, brent['close'], color='#888', linewidth=0.8, alpha=0.5, zorder=1)
for r in range(3):
    mask = regime_series == r
    label = regime_labels[r]
    dates_regime = feat.index[mask]
    prices_regime = brent['close'].reindex(dates_regime)
    ax.scatter(dates_regime, prices_regime, c=colors[label], s=3,
               label=label, alpha=0.6, zorder=2)
ax.set_title('Market Regime Detection (K-Means, 3 Regimes)')
ax.set_ylabel('Brent Price ($/bbl)')
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig('../docs/images/market_regimes.png', bbox_inches='tight')
plt.show()
for r, label in regime_labels.items():
    cnt = (regime_series == r).sum()
    print(f"  {label:<10} {cnt:>5} days ({cnt/len(regime_series)*100:.1f}%)")

## 6. Macro Factor Correlations

In [ ]:
# Build a correlation matrix between oil and macro factors
from src.features.macro_features import load_macro_series

macro = load_macro_series(brent.index)

# Daily returns for correlation
corr_data = pd.DataFrame({'brent_return': returns})
for col in ['dxy', 'gold', 'yield_10y']:
    if col in macro.columns:
        corr_data[f'{col}_return'] = macro[col].pct_change()

corr_data = corr_data.dropna()
corr_matrix = corr_data.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, ax=axes[0], linewidths=0.3)
axes[0].set_title('Cross-Asset Return Correlation')

# Rolling correlation: oil vs DXY
if 'dxy_return' in corr_data.columns:
    roll_corr = corr_data['brent_return'].rolling(60).corr(corr_data['dxy_return'])
    axes[1].plot(roll_corr.index, roll_corr, color='#E24B4A', linewidth=1.0)
    axes[1].axhline(0, color='#888', linewidth=0.8)
    axes[1].axhline(roll_corr.mean(), color='#378ADD', linewidth=1.2,
                    linestyle='--', label=f'Mean: {roll_corr.mean():.3f}')
    axes[1].set_title('Rolling Brent–DXY Correlation (60-day)')
    axes[1].set_ylabel('Correlation')
    axes[1].legend()
    axes[1].set_ylim(-1, 1)

plt.tight_layout()
plt.savefig('../docs/images/macro_correlation.png', bbox_inches='tight')
plt.show()
print("Key insight: Brent-DXY correlation is typically negative (stronger USD = cheaper oil in USD terms)")

## Key Takeaways
- Oil prices exhibit **fat-tailed returns** — linear models underestimate tail risk
- Three clear **market regimes** exist: bull, bear, sideways (each requires different model behavior)
- **USD (DXY) is the strongest macro driver** — negative correlation with oil prices
- **Seasonality** is present but weak vs. supply shocks
- Next step: → `02_feature_engineering.ipynb`